In [1]:
import awkward as ak
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from lightning import LightningDataModule
from vbf_tagger.tools.data.general import initialize_p4

import vector
import tqdm
import os
import glob

from torch import nn
from torch.utils.data import Dataset
import torch.nn.functional as F

import sklearn
import sklearn.metrics
import matplotlib
import matplotlib.pyplot as plt

from omegaconf import OmegaConf
from omegaconf import DictConfig

from torch.utils.data import IterableDataset
from collections.abc import Sequence

import vbf_tagger.tools.data.general as g
from hydra import compose, initialize

from torch.utils.data._utils.collate import default_collate

/opt/conda/lib/python3.11/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
def load_parquet(input_path: str, columns: list = None) -> ak.Array:
    """ Loads the contents of the .parquet file specified by the input_path
    Args:
        input_path : str
            The path to the .parquet file to be loaded.
        columns : list
            Names of the columns/branches to be loaded from the .parquet file
    Returns:
        input_data : ak.Array
            The data from the .parquet file
    """
    ret = ak.from_parquet(input_path, columns=columns)
    ret = ak.Array({k: ret[k] for k in ret.fields})
    return ret

def load_all_data(input_loc: str, n_files: int = None, columns: list = None) -> ak.Array:
    """Loads all .parquet files specified by the input. The input can be a list of input_paths, a directory where the files
    are located or a wildcard path.
    Args:
        input_loc : str
            Location of the .parquet files.
        n_files : int
            [default: None] Maximum number of input files to be loaded. By default all will be loaded.
        columns : list
            [default: None] Names of the columns/branches to be loaded from the .parquet file. By default all columns will
            be loaded
    Returns:
        input_data : ak.Array
            The concatenated data from all the loaded files
    """
    if n_files == -1:
        n_files = None
    if isinstance(input_loc, list):
        input_files = input_loc[:n_files]
    elif isinstance(input_loc, str):
        if os.path.isdir(input_loc):
            input_loc = os.path.expandvars(input_loc)
            input_files = glob.glob(os.path.join(input_loc, "*.parquet"))[:n_files]
        elif "*" in input_loc:
            input_files = glob.glob(input_loc)[:n_files]
        else:
            raise ValueError(f"Unexpected input_loc")
    else:
        raise ValueError(f"Unexpected input_loc")
    input_data = []
    for i, file_path in enumerate(input_files):
        print(f"[{i+1}/{len(input_files)}] Loading from {file_path}")
        try:
            input_data.append(load_parquet(file_path, columns=columns))
        except ValueError:
            print(f"{file_path} does not exist")
    if len(input_data) > 0:
        data = ak.concatenate(input_data)
        print("Input data loaded")
    else:
        raise ValueError(f"No files found in {input_loc}")
    return data


In [3]:
vbfhh = load_all_data("/home/norman/samples/hh_vbf/")

[1/7] Loading from /home/norman/samples/hh_vbf/hh_vbf_hbb_htt_kv1_k2v0_kl1_madgraph_events_0.parquet
[2/7] Loading from /home/norman/samples/hh_vbf/hh_vbf_hbb_htt_kv1_k2v2_kl1_madgraph_events_0.parquet
[3/7] Loading from /home/norman/samples/hh_vbf/hh_vbf_hbb_htt_kv1_k2v0_kl1_madgraph_events_1.parquet
[4/7] Loading from /home/norman/samples/hh_vbf/hh_vbf_hbb_htt_kv1_k2v1_kl2_madgraph_events_0.parquet
[5/7] Loading from /home/norman/samples/hh_vbf/hh_vbf_hbb_htt_kv1_k2v0_kl1_madgraph_events_2.parquet
[6/7] Loading from /home/norman/samples/hh_vbf/hh_vbf_hbb_htt_kv1_k2v1_kl1_madgraph_events_1.parquet
[7/7] Loading from /home/norman/samples/hh_vbf/hh_vbf_hbb_htt_kv1_k2v1_kl1_madgraph_events_0.parquet
Input data loaded


In [80]:
def _compute_cartesian(pt, eta, phi):
    """
    Compute px, py, pz from (pt, eta, phi).
    px = pt * cos(phi)
    py = pt * sin(phi)
    pz = pt * sinh(eta)
    Returns shape (..., 3)
    """
    px = pt * np.cos(phi)
    py = pt * np.sin(phi)
    pz = pt * np.sinh(eta)
    return np.stack([px, py, pz], axis=-1)

In [112]:
def build_jet_tensors(data: ak.Array, features: list, max_jets: int = 14):
    """
    Build padded (n_events, max_jets, n_features) jet tensors for jet-level training.

    Returns:
        X:       float32 (B, P, F)
        coords:  float32 (B, P, 4)  # px, py, pz, E
        mask:    bool    (B, P)
        labels:  float32 (B, P)
    """

    jets_raw = data.TrainingJet
    jets_p4  = initialize_p4(data.TrainingJet)

    # ---------------------------
    # 1. Sort by descending pt
    # ---------------------------
    order = ak.argsort(jets_p4.pt, ascending=False)
    jets_raw = jets_raw[order]
    jets_p4  = jets_p4[order]

    # ---------------------------
    # padding helper
    # ---------------------------
    def pad(field):
        arr = ak.pad_none(field, max_jets, clip=True)
        arr = ak.fill_none(arr, 0)
        return ak.to_numpy(arr)

    # ---------------------------
    # 2. Derived safe features
    # ---------------------------
    pt_arr   = pad(jets_p4.pt).astype(np.float32)
    eta_arr  = pad(jets_p4.eta).astype(np.float32)
    phi_arr  = pad(jets_p4.phi).astype(np.float32)
    mass_arr = pad(jets_p4.mass).astype(np.float32)

    pt_safe   = np.clip(pt_arr,   1e-6, None)
    mass_safe = np.clip(mass_arr, 1e-6, None)

    derived = {
        "log_pt":   np.log(pt_safe),
        "sin_phi":  np.sin(phi_arr),
        "cos_phi":  np.cos(phi_arr),
        "log_mass": np.log(mass_safe),
    }

    # ---------------------------
    # 3. Build X
    # ---------------------------
    feat_list = []
    for f in features:
        if f in derived:
            arr = derived[f]
        elif hasattr(jets_p4, f):
            arr = pad(getattr(jets_p4, f))
        elif hasattr(jets_raw, f):
            arr = pad(getattr(jets_raw, f))
        else:
            raise ValueError(f"Feature '{f}' not found.")
        feat_list.append(arr[..., None])

    X = np.concatenate(feat_list, axis=-1).astype(np.float32)

    # ---------------------------
    # 4. Mask
    # ---------------------------
    pt_orig = pad(initialize_p4(data.TrainingJet).pt[order])
    mask = (pt_orig > 0)

    # ---------------------------
    # 5. Labels
    # ---------------------------
    labels = pad(jets_raw.isVBF).astype(np.float32)

    # ---------------------------
    # 6. Coords: (px,py,pz,E)
    # ---------------------------
    xyz3 = _compute_cartesian(pt_arr, eta_arr, phi_arr)  # (B,P,3)

    energy = np.sqrt(
        xyz3[..., 0]**2 + xyz3[..., 1]**2 + xyz3[..., 2]**2 + mass_safe**2
    )

    coords = np.concatenate([xyz3, energy[..., None]], axis=-1).astype(np.float32)
    # (B, P, 4)

    return X, coords, mask.astype(bool), labels


In [113]:
features = [
    "log_pt",
    "eta",
    "sin_phi",
    "cos_phi",
    "log_mass",
    "btagDeepFlavB",
    "btagDeepFlavQG",
    "btagPNetB",
    "btagPNetQvG",
    "btagPNetTauVJet",
    "hhbtag",
]

In [147]:
x, c, m, y = build_jet_tensors(vbfhh, features)

In [148]:
print(x.shape)
print(c.shape)
print(m.shape)
print(y.shape)

(203006, 14, 11)
(203006, 14, 4)
(203006, 14)
(203006, 14)


In [150]:
xt = x.transpose(0, 1, 2).contiguous()

AttributeError: 'numpy.ndarray' object has no attribute 'contiguous'

In [141]:
xt.shape

(203006, 11, 14)

In [118]:
class JetDataModule(LightningDataModule):
    """
    Lightning DataModule producing batches as dicts compatible with ParticleTransformer.

    Each batch is a dict:
        {
          "points": Tensor[B, N, F],
          "points_mask": BoolTensor[B, N],
          "points_xyz": Tensor[B, N, 3],
          "labels": Tensor[B, N]
        }
    """
    def __init__(self, cfg: DictConfig, features: list = None, max_jets: int = 14):
        super().__init__()
        self.cfg = cfg
        self.features = features
        self.max_jets = max_jets
        self.batch_size = cfg.training.dataloader.batch_size
        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None
        # scaler (applies to features only)
        self.mean = None
        self.std = None
        self.pos_weight = 1.0

    def _get_files(self, dataset_keys, split_dir):
        dataset_paths = []
        for key in dataset_keys:
            base = self.cfg.dataset.datasets[key]
            path = os.path.join(base, split_dir)
            files = glob.glob(os.path.join(path, "*.parquet"))
            dataset_paths.extend(files)
        return sorted(dataset_paths)

    def setup(self, stage=None):
        # train/val
        if stage == "fit" or stage is None:
            train_files = self._get_files(self.cfg.dataset.train_dataset, self.cfg.dataset.train_dir)
            val_files   = self._get_files(self.cfg.dataset.val_dataset, self.cfg.dataset.val_dir)
            print(f" Found {len(train_files)} train files, {len(val_files)} val files")

            data_train = ak.from_parquet(train_files)
            X_train, coords_train, mask_train, y_train = build_jet_tensors(data_train, self.features, max_jets=self.max_jets)

            data_val = ak.from_parquet(val_files)
            X_val, coords_val, mask_val, y_val = build_jet_tensors(data_val, self.features, max_jets=self.max_jets)

            # compute scaler from TRAIN only (only real jets)
            mask_flat = mask_train.reshape(-1)
            X_flat = X_train.reshape(-1, X_train.shape[-1])
            valid = mask_flat
            if valid.sum() == 0:
                raise RuntimeError("No valid jets in training data.")
            mean = X_flat[valid].mean(axis=0)
            std = X_flat[valid].std(axis=0) + 1e-6
            self.mean = mean
            self.std = std

            # Save scaler for later inference
            scaler_path = os.path.join(self.cfg.training.models_dir, "scaler.npz")
            os.makedirs(self.cfg.training.models_dir, exist_ok=True)
            np.savez(scaler_path, mean=mean, std=std)
            print(f" Saved scaler → {scaler_path}")

            # apply normalization ONLY to X (features). coords left unchanged.
            X_train = (X_train - mean[None, None, :]) / std[None, None, :]
            X_val   = (X_val   - mean[None, None, :]) / std[None, None, :]

            # compute class weight (pos_weight = n_neg / n_pos) using TRAIN labels only
            n_pos = (y_train == 1).sum()
            n_neg = (y_train == 0).sum()
            self.pos_weight = float(n_neg) / max(1.0, float(n_pos))
            print(f" Class balance (train): {n_pos} positives, {n_neg} negatives → pos_weight={self.pos_weight:.2f}")

            # convert to tensors and datasets (event-wise)
            Xt = torch.tensor(X_train, dtype=torch.float32)          # (n_events, N, F)
            coords_t = torch.tensor(coords_train, dtype=torch.float32)  # (n_events, N, 3)
            mt = torch.tensor(mask_train, dtype=torch.bool)
            yt = torch.tensor(y_train, dtype=torch.float32)

            Xv = torch.tensor(X_val, dtype=torch.float32)
            coords_v = torch.tensor(coords_val, dtype=torch.float32)
            mv = torch.tensor(mask_val, dtype=torch.bool)
            yv = torch.tensor(y_val, dtype=torch.float32)

            # Save datasets as TensorDataset of (points, coords, mask, labels)
            self.train_dataset = TensorDataset(Xt, coords_t, mt, yt)
            self.val_dataset   = TensorDataset(Xv, coords_v, mv, yv)

        if stage == "test" or stage is None:
            # Load scaler if available
            scaler_path = os.path.join(self.cfg.training.models_dir, "scaler.npz")
            if os.path.exists(scaler_path):
                scaler = np.load(scaler_path)
                self.mean = scaler["mean"]
                self.std = scaler["std"]
                print(f" Loaded scaler from {scaler_path}")
            else:
                raise RuntimeError(f"Scaler not found at {scaler_path} — cannot normalize test data.")

            test_files = self._get_files(self.cfg.dataset.test_dataset, self.cfg.dataset.test_dir)
            print(f" Found {len(test_files)} test files")
            data_test = ak.from_parquet(test_files)
            X_test, coords_test, mask_test, y_test = build_jet_tensors(data_test, self.features, max_jets=self.max_jets)
            X_test = (X_test - self.mean[None, None, :]) / self.std[None, None, :]
            Xt = torch.tensor(X_test, dtype=torch.float32)
            coords_t = torch.tensor(coords_test, dtype=torch.float32)
            mt = torch.tensor(mask_test, dtype=torch.bool)
            yt = torch.tensor(y_test, dtype=torch.float32)
            self.test_dataset = TensorDataset(Xt, coords_t, mt, yt)

    @staticmethod
    def _collate_to_dict(batch):
        """
        Convert a batch (list of tuples) to dict:
          - uses default_collate to stack per-field
          - returns: {"points","points_xyz","points_mask","labels"}
        """
        # default_collate expects list inputs, so use it directly
        collated = default_collate(batch)  # returns tuple stacked tensors
        # collated is a tuple: (points, coords, mask, labels)
        points, coords, mask, labels = collated
        mask = mask.unsqueeze(-1) 
        return {
            "points": points,            # (B, N, F)
            "points_xyz": coords,        # (B, N, 3)
            "points_mask": mask,         # (B, N)
            "labels": labels,            # (B, N)
        }

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=4,
            collate_fn=self._collate_to_dict,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=4,
            collate_fn=self._collate_to_dict,
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=4,
            collate_fn=self._collate_to_dict,
        )

In [119]:
with initialize(version_base=None, config_path="vbf_tagger/config/", job_name="compare"):
    cfg = compose(config_name="main")

In [120]:
dm = JetDataModule(cfg, features, max_jets=14)

In [121]:
dm.setup("fit")

 Found 14 train files, 14 val files
 Saved scaler → None/classification/models/scaler.npz
 Class balance (train): 387801 positives, 3738825 negatives → pos_weight=9.64


In [122]:
train_loader = dm.train_dataloader()

In [123]:
train_loader

In [124]:
for batch in train_loader:
    print("Batch type:", type(batch))
    print("Batch length:", len(batch))
    print("Batch:", batch)
    break

Batch type: <class 'dict'>
Batch length: 4
Batch: {'points': tensor([[[ 1.9939e+00, -5.7580e-02, -8.8640e-01,  ...,  3.6470e-01,
           5.8106e-01,  7.2376e-01],
         [-6.8600e-02,  5.4852e-02,  1.0617e-01,  ...,  3.3966e-01,
           5.8275e-01,  7.2376e-01],
         [-9.8841e-02,  2.1012e+00, -1.4069e+00,  ...,  2.9197e-01,
          -1.4800e+00, -1.3892e+00],
         ...,
         [-2.0817e+01,  5.9423e-03,  5.7531e-04,  ...,  1.4510e-01,
           5.7968e-01,  7.2374e-01],
         [-2.0817e+01,  5.9423e-03,  5.7531e-04,  ...,  1.4510e-01,
           5.7968e-01,  7.2374e-01],
         [-2.0817e+01,  5.9423e-03,  5.7531e-04,  ...,  1.4510e-01,
           5.7968e-01,  7.2374e-01]],

        [[ 1.1942e+00,  5.7961e-01,  6.8857e-01,  ...,  3.7705e-01,
           5.8035e-01,  7.2376e-01],
         [-1.0481e-01, -1.7979e-01, -9.7149e-01,  ...,  3.7524e-01,
           5.8025e-01,  7.2375e-01],
         [-3.2269e-01, -9.8608e-01, -1.4167e+00,  ...,  2.3011e-01,
           5.80

In [158]:
x_ = batch["points"]
coords = batch["points_xyz"]
mask = batch["points_mask"]

In [159]:
print(x_.shape)
print(coords.shape)
print(mask.shape)

torch.Size([256, 14, 11])
torch.Size([256, 14, 4])
torch.Size([256, 14, 1])


In [160]:
x_ = x_.transpose(1, 2)     # (B, P, C) → (B, C, P)
coords = coords.transpose(1, 2)  # (B, P, 4) → (B, 4, P)

In [161]:
print(x_.shape)
print(coords.shape)

torch.Size([256, 11, 14])
torch.Size([256, 4, 14])


In [162]:
vbfhh.TrainingJet.pt

<Array [[300, 103, 58.7, 55.7, 15.9], ..., [...]] type='203006 * var * float32'>

In [163]:
xt_ = x_.transpose(1, 2).contiguous()

In [164]:
xt_.shape

torch.Size([256, 14, 11])

In [165]:
x_.shape

torch.Size([256, 11, 14])

In [168]:
mask_.shape

torch.Size([256, 1, 14, 1])